# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Priyansh-rath18/flyrank-internship-/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**One row = one content page (`content_id`), nested under one client (`client_id`), as a single point-in-time snapshot — not a repeated time series.** Every metric is a trailing window ending at the export date: `impressions_90d`, `clicks_90d`, `sessions_90d` and friends cover the trailing 90 days, and the trend inputs split the most recent 60 of those 90 days into two 30-day halves — `impressions_last_30d`/`clicks_last_30d`/`sessions_last_30d` (days 1-30 back) vs `impressions_prev_30d`/`clicks_prev_30d`/`sessions_prev_30d` (days 31-60 back). `content_age_days` confirms every row is already >= 90 days old, matching the 90-day window. There is no shared calendar date column — the window is relative to each row's own export time, so different rows are not guaranteed to share the same absolute date range.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd

url = "https://raw.githubusercontent.com/Priyansh-rath18/flyrank-internship-/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

print("rows x cols:", df.shape)
print("unique content_id:", df["content_id"].nunique(), "(grain check: should equal row count)")
print("unique client_id:", df["client_id"].nunique())
print("content_age_days min:", df["content_age_days"].min(), "(should be >= 90, matching the 90-day window)")

dupe_ids = df["content_id"].value_counts()
print("content_ids appearing more than once:", int((dupe_ids > 1).sum()))

rows x cols: (30000, 44)
unique content_id: 30000 (grain check: should equal row count)
unique client_id: 32
content_age_days min: 90 (should be >= 90, matching the 90-day window)
content_ids appearing more than once: 0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Feature** (knowable before the moment we'd score a page): `search_volume`, `competition`, `competition_level`, `cpc`, `content_type`, `main_intent`, `word_count`, `char_count`, `content_age_days`, `age_tier`, `age_tier_order`, `days_since_last_update`, `freshness_tier`, `word_count_tier`, `char_count_tier`, `impressions_90d`, `clicks_90d`, `pageviews_90d`, `sessions_90d`, `users_90d`, `engaged_sessions_90d`, `ai_sessions_90d`, `scroll_events_90d`, `days_with_impressions`, `days_with_sessions`, `clicks_last_30d`, `sessions_last_30d`, `clicks_prev_30d`, `sessions_prev_30d`, `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, `impression_tier`, `position_tier`.

**Label / proxy**: `trend_direction` and `trend_pct` — the pipeline defines the decline label directly from these. `impressions_last_30d` and `impressions_prev_30d` are also label bucket, not feature bucket: they are the literal inputs to the `trend_pct` formula, so using them as features would let the model reconstruct the label almost exactly (near-perfect leakage), even though the data dictionary only calls out `trend_direction`/`trend_pct` by name.

**Context** (grouping/joining/splitting only, never features): `content_id` (unique per row, pseudonym), `client_id` (32 distinct pseudonyms — use for client-holdout splits so no client leaks across train/test).

**Excluded**: `provider_used` and `model_used` — the dictionary marks both "not a model feature"; they describe which LLM/vendor generated the article, an operational/product detail rather than a signal an editor could act on, and could quietly encode a vendor-quality artifact instead of a genuine decline pattern.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

feature_cols = ["search_volume","competition","competition_level","cpc","content_type","main_intent",
"word_count","char_count","content_age_days","age_tier","age_tier_order",
"days_since_last_update","freshness_tier","word_count_tier","char_count_tier",
"impressions_90d","clicks_90d","pageviews_90d","sessions_90d","users_90d",
"engaged_sessions_90d","ai_sessions_90d","scroll_events_90d","days_with_impressions",
"days_with_sessions","clicks_last_30d","sessions_last_30d","clicks_prev_30d",
"sessions_prev_30d","ctr","avg_position","engagement_rate","scroll_rate",
"ai_traffic_pct","impression_tier","position_tier"]
label_cols = ["trend_direction", "trend_pct", "impressions_last_30d", "impressions_prev_30d"]
context_cols = ["content_id", "client_id"]
excluded_cols = ["provider_used", "model_used"]

all_buckets = feature_cols + label_cols + context_cols + excluded_cols
print("columns covered:", len(all_buckets), "of", df.shape[1])
print("columns missing from any bucket:", set(df.columns) - set(all_buckets))

valid = df["impressions_prev_30d"] != 0
recomputed = (df.loc[valid, "impressions_last_30d"] - df.loc[valid, "impressions_prev_30d"]) / df.loc[valid, "impressions_prev_30d"] * 100
close_share = (recomputed.round(1) - df.loc[valid, "trend_pct"]).abs().le(0.15).mean()
print("share of non-blank rows where impressions_last_30d/prev_30d alone reconstruct trend_pct within 0.15pp:", round(close_share, 3))

print(df[["provider_used", "model_used"]].isna().mean())

columns covered: 44 of 44
columns missing from any bucket: set()
share of non-blank rows where impressions_last_30d/prev_30d alone reconstruct trend_pct within 0.15pp: 1.0
provider_used    0.7146
model_used       0.1911
dtype: float64


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Four claims from sections 1-2, each checked in the code cell below: (1) **grain** — `content_id` is unique per row, so "one row = one content page" holds; (2) **counts** — 30,000 rows span 32 distinct `client_id` values, matching the dictionary; (3) **missingness** — keyword-context columns (`search_volume`, `competition`, `cpc`) and content-length columns (`word_count`, `char_count`) go blank together along `content_type` lines, not randomly, so a blind `fillna(0)` would silently encode content type; (4) **windows** — `content_age_days` is always >= 90 (matches the 90-day window) and `impressions_prev_30d == 0` for exactly the rows where `trend_pct` is blank, matching the documented 3,388-row footnote.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

grain_check = df["content_id"].value_counts()
print("rows with duplicate content_id (should be 0 if grain holds):", int((grain_check > 1).sum()))

print("total rows:", len(df))
print("distinct clients:", df["client_id"].nunique())
print(df.groupby("client_id").size().describe())

missing_by_type = df.groupby("content_type")[["search_volume", "word_count"]].apply(lambda g: g.isna().mean())
print(missing_by_type)

print("rows with content_age_days < 90 (should be 0):", int((df["content_age_days"] < 90).sum()))
blank_trend = int(df["trend_pct"].isna().sum())
zero_prev = int((df["impressions_prev_30d"] == 0).sum())
print("blank trend_pct rows:", blank_trend, "| impressions_prev_30d == 0 rows:", zero_prev)

rows with duplicate content_id (should be 0 if grain holds): 0
total rows: 30000
distinct clients: 32
count      32.000000
mean      937.500000
std      1376.387113
min         3.000000
25%       110.250000
50%       567.000000
75%      1058.750000
max      7008.000000
dtype: float64
                    search_volume  word_count
content_type                                 
comparison article       0.000000    0.000000
feedly article           1.000000    0.000000
keyword article          0.013673    0.282979
rows with content_age_days < 90 (should be 0): 0
blank trend_pct rows: 3388 | impressions_prev_30d == 0 rows: 3388


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This 30k-row starter slice is a **single aggregated snapshot, not a time series**: it cannot show what happened on any individual day, cannot prove that a specific edit caused a page's traffic to recover, and cannot support any claim stronger than "directionally associated with an observed decline" — correlation only, never causation. It also cannot generalize past the 32 clients and one export moment it was pulled from; the full warehouse (`fact_content_daily_performance`, ~79M rows across ~17 months) is needed for anything about seasonality, longer trends, or across-client comparisons, and that panel is itself unbalanced — per-client history depth differs (`dim_clients.gsc_data_start`), a third of clients have little usable history, and rows before a client's `ga4_data_start` are GSC-only with GA4 columns zero-filled (flagged `ga4_data_available`, which can also be NULL and must not be treated as FALSE). Any label defined on a page's most recent window also overlaps `fact_content_query_90d`'s fixed 90-day window — only `*_prev30`-style columns are safe features there, so this data can never honestly answer a question about the exact window it was used to build the label from.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

date_like_cols = [c for c in df.columns if "date" in c.lower()]
print("columns whose name contains 'date':", date_like_cols, "-- this is a day-count field (days since update), not a calendar date; the starter CSV has no calendar-date column at all")

print("clients covered by this slice:", df["client_id"].nunique(), "-- claims only generalize to these clients/this export moment")

nested_check = int((df["clicks_last_30d"] + df["clicks_prev_30d"] > df["clicks_90d"]).sum())
print("rows where last30+prev30 clicks exceed the 90d total (should be ~0, confirms the 30d windows nest inside the 90d window, not independent of it):", nested_check)

columns whose name contains 'date': ['days_since_last_update'] -- this is a day-count field (days since update), not a calendar date; the starter CSV has no calendar-date column at all
clients covered by this slice: 32 -- claims only generalize to these clients/this export moment
rows where last30+prev30 clicks exceed the 90d total (should be ~0, confirms the 30d windows nest inside the 90d window, not independent of it): 0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.